# 🧠 Gmail AI Assistant: Model Exploration and Training

Welcome to the interactive exploration notebook for the **Gmail AI Assistant**. This notebook demonstrates how the extractive summarization model is built, trained, and used for generating summaries from emails.

## 📖 Project Overview
The assistant uses an **Extractive Summarization** approach. Unlike abstractive summarization (which generates new text), extractive summarization identifies and extracts the most important sentences from the original document.

### 🏗️ Architecture
The model consists of:
1.  **Sentence Encoder**: A Bidirectional LSTM that converts a sequence of word embeddings into a fixed-size sentence vector.
2.  **Document context layer**: A second Bidirectional LSTM that processes the sequence of sentence vectors to understand the context of each sentence within the whole document.
3.  **Classification Head**: A simple feed-forward network that outputs a probability (0 to 1) for each sentence, representing its importance.

### 🛠️ Setup and Imports
First, let's set up the environment and import our refactored components.

In [1]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

# Add the src directory to the path so we can import our modules
sys.path.append(os.path.abspath('../'))

from src.gmail_assistant.ml.data import create_synthetic_dataset, generate_synthetic_email
from src.gmail_assistant.ml.preprocessor import CustomTokenizer, Vocabulary, split_sentences, pad_sequence
from src.gmail_assistant.ml.model import ExtractiveSummarizer
from src.gmail_assistant.ml.inference import Summarizer

print("Environment setup complete! ✅")

ModuleNotFoundError: No module named 'torch'

## 📊 Data Exploration
Since we don't have a massive labeled dataset of emails, we use a **synthetic data generator**. This generator creates emails with varying structures and assigns labels to sentences based on the presence of "important" keywords (like *deadline*, *urgent*, *invoice*).

In [ ]:
# Generate a single synthetic email for demonstration
email_text, labels = generate_synthetic_email()
sentences = split_sentences(email_text)

print(f"Generated Email Text:\n{'-' * 20}")
for i, (sent, label) in enumerate(zip(sentences, labels)):
    importance = "⭐ [IMPORTANT]" if label == 1 else "   [FILLER]"
    print(f"{i+1}. {importance} {sent}")

## 🧪 Model Inference
Let's see how our trained model performs on a test email. We'll use the `Summarizer` class, which handles all the preprocessing and inference details.

In [ ]:
# Define a test email
test_email = """
Subject: Urgent: Q1 Financial Review Meeting
I hope you are having a productive week.
We need to finalize the Q1 financial report by this Friday.
There will be a mandatory meeting tomorrow at 10 AM to discuss the results.
Please bring your departmental updates and confirm your attendance.
Looking forward to seeing you there.
Best regards, Finance Team.
"""

try:
    # Initialize the summarizer (it loads the model from models/model.pth by default)
    summarizer = Summarizer(model_path='../models/model.pth')
    
    # Generate a summary with the top 3 sentences
    summary = summarizer.summarize(test_email, top_n=3)
    
    print(f"\nOriginal Email:\n{test_email}")
    print(f"{'-' * 40}")
    print(f"Generated Summary:\n{summary}")
except FileNotFoundError:
    print("❌ Error: Model checkpoint not found. Please train the model first using 'python main.py train'.")

## 📈 Visualization of Sentence Scores
Let's peek inside the model and visualize the scores it assigns to each sentence in our test email.

In [ ]:
def visualize_scores(text, summarizer):
    if summarizer is None or not hasattr(summarizer, 'model'):
        print("❌ Error: Summarizer is not properly initialized. Please run the inference cell first.")
        return

    sentences = split_sentences(text)
    encoded_sentences = []
    for sent in sentences:
        tokens = summarizer.tokenizer.tokenize(sent)
        indices = summarizer.vocab.encode(tokens)
        padded = pad_sequence(indices, summarizer.max_sent_len)
        encoded_sentences.append(padded)
    
    sentences_tensor = torch.tensor(encoded_sentences, dtype=torch.long)
    
    with torch.no_grad():
        scores = summarizer.model([sentences_tensor])[0].cpu().numpy()
    
    # Plotting
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(sentences)), scores, color='skyblue')
    plt.yticks(range(len(sentences)), [s[:50] + '...' if len(s) > 50 else s for s in sentences])
    plt.xlabel('Importance Score')
    plt.title('Sentence Importance Scores assigned by AI Model')
    plt.gca().invert_yaxis()
    plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold')
    plt.legend()
    plt.tight_layout()
    plt.show()

try:
    visualize_scores(test_email, summarizer)
except NameError as e:
    print(f"❌ Error: '{e.name}' is not defined. Ensure you have run all previous cells.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")